# STEP 3 — Directed Deep Graph Infomax (DGI) for Cora



In [ ]:
# ------------------------------------------------------------
# 0. Imports
# ------------------------------------------------------------

import copy
import json
import os
import pickle
import random
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from tqdm.auto import tqdm

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.utils import from_networkx

In [ ]:
# ------------------------------------------------------------
# 1. Reproducibility and device
# ------------------------------------------------------------

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_device():
    return torch.device(
        "cuda" if torch.cuda.is_available() else "cpu"
    )


set_seed(123)

device = get_device()

print("Device:", device)

In [ ]:
# ------------------------------------------------------------
# 2. Input and output paths
# ------------------------------------------------------------

EXPECTED_GRAPH_FILE = "graphs_directed_adamic_adar_nsum.pkl"

preferred_graph_path = (
    Path("cora_directed_link_prediction")
    / EXPECTED_GRAPH_FILE
)

if preferred_graph_path.exists():
    graphs_path = preferred_graph_path
else:
    matching_paths = list(
        Path(".").rglob(EXPECTED_GRAPH_FILE)
    )

    if len(matching_paths) == 0:
        available_pkl_files = list(
            Path(".").rglob("*.pkl")
        )

        print("Current working directory:")
        print(Path.cwd())

        print("\nAvailable .pkl files:")
        for path in available_pkl_files:
            print(path)

        raise FileNotFoundError(
            f"Could not find '{EXPECTED_GRAPH_FILE}'. "
            "Run the corrected directed Step 2 notebook first."
        )

    if len(matching_paths) > 1:
        print("Multiple matching files were found:")

        for i, path in enumerate(matching_paths):
            print(f"{i}: {path}")

        raise ValueError(
            "Set graphs_path manually to the correct Step 2 output."
        )

    graphs_path = matching_paths[0]


ard_path = Path(
    "ard_matrix_cora_directed_out.csv"
)

# Standardized edge list produced by the corrected Step 0:
# source = citing paper, target = cited paper.
original_edges_path = Path(
    "cora_directed_edges.csv"
)

labels_path = Path(
    "cora.node_labels"
)

output_folder = Path(
    "cora_directed_dgi_ard"
)

output_folder.mkdir(
    parents=True,
    exist_ok=True
)

print("Directed synthetic graphs:", graphs_path)
print("Outgoing ARD:", ard_path)
print("Original directed graph:", original_edges_path)
print("Citation orientation: citing -> cited")
print("Output folder:", output_folder)


In [ ]:
# ------------------------------------------------------------
# 3. Load outgoing ARD features
# ------------------------------------------------------------

def load_outgoing_ard_features(ard_path):
    features_df = pd.read_csv(
        ard_path,
        index_col=0
    )

    features_df.index = (
        features_df.index.astype(int)
    )

    features_df.index.name = "node_id"

    if features_df.index.duplicated().any():
        raise ValueError(
            "The ARD matrix contains duplicated node IDs."
        )

    if features_df.isna().any().any():
        raise ValueError(
            "The ARD matrix contains missing values."
        )

    if (features_df.to_numpy() < 0).any():
        raise ValueError(
            "The ARD matrix contains negative values."
        )

    features_array = (
        features_df
        .to_numpy(dtype=np.float32, copy=True)
    )

    features_tensor = torch.from_numpy(
        features_array
    )

    return features_df, features_tensor

In [ ]:
# ------------------------------------------------------------
# 4. Load directed graphs
# ------------------------------------------------------------

def load_augmented_directed_graphs(graphs_path):
    with open(graphs_path, "rb") as file:
        graphs = pickle.load(file)

    if not isinstance(graphs, (list, tuple)):
        raise TypeError(
            "The Step 2 pickle must contain a list of graphs."
        )

    graphs = list(graphs)

    if len(graphs) == 0:
        raise ValueError(
            "The Step 2 graph list is empty."
        )

    for graph_id, graph in enumerate(graphs, start=1):
        if not isinstance(graph, nx.DiGraph):
            raise TypeError(
                f"Graph {graph_id} is not a networkx.DiGraph."
            )

    return graphs


def load_original_cora_directed_graph(
    edges_path,
    labels_path,
    node_order
):
    """
    Load the standardized directed edge list generated by Step 0.

    Expected columns:
        source, target, weight

    with:
        source = citing paper
        target = cited paper

    Thus every edge is already oriented as:
        citing -> cited
    """

    edges_df = pd.read_csv(
        edges_path,
        sep=",",
        header=None,
        names=["source", "target", "weight"]
    )

    edges_df["source"] = (
        edges_df["source"].astype(int)
    )

    edges_df["target"] = (
        edges_df["target"].astype(int)
    )

    edges_df = (
        edges_df[
            edges_df["source"]
            != edges_df["target"]
        ]
        .drop_duplicates(
            subset=["source", "target"]
        )
        .reset_index(drop=True)
    )

    labels_df = pd.read_csv(
        labels_path,
        sep=",",
        header=None,
        names=["node", "label"]
    )

    labels_df["node"] = (
        labels_df["node"].astype(int)
    )

    labelled_nodes = set(
        labels_df["node"].tolist()
    )

    if labelled_nodes != set(node_order):
        raise ValueError(
            "The labels and outgoing ARD contain different nodes."
        )

    graph = nx.DiGraph()

    # Insert nodes in exactly the same order as the ARD rows.
    graph.add_nodes_from(
        list(map(int, node_order))
    )

    graph.add_edges_from(
        edges_df[
            ["source", "target"]
        ].itertuples(
            index=False,
            name=None
        )
    )

    print(
        "Original Cora graph loaded with orientation: "
        "citing -> cited"
    )

    return graph


In [ ]:
# ------------------------------------------------------------
# 5. Preserve node order and validate graph alignment
# ------------------------------------------------------------

def enforce_directed_node_order(
    graph,
    node_order
):
    if not graph.is_directed():
        raise TypeError(
            "A directed graph is required."
        )

    node_order = list(map(int, node_order))

    if set(graph.nodes()) != set(node_order):
        missing = (
            set(node_order)
            - set(graph.nodes())
        )

        extra = (
            set(graph.nodes())
            - set(node_order)
        )

        raise ValueError(
            "Graph and ARD contain different nodes. "
            f"Missing: {sorted(missing)[:10]}; "
            f"Extra: {sorted(extra)[:10]}"
        )

    ordered_graph = nx.DiGraph()

    ordered_graph.add_nodes_from(
        node_order
    )

    ordered_graph.add_edges_from(
        graph.edges()
    )

    return ordered_graph


def enforce_directed_node_order_all(
    graphs,
    node_order
):
    return [
        enforce_directed_node_order(
            graph,
            node_order
        )
        for graph in graphs
    ]


def check_graph_feature_alignment(
    graph,
    features_df,
    features_tensor
):
    if not graph.is_directed():
        raise TypeError(
            "The graph must remain directed."
        )

    if nx.number_of_selfloops(graph) > 0:
        raise ValueError(
            "The graph contains self-loops."
        )

    if graph.number_of_nodes() != features_tensor.shape[0]:
        raise ValueError(
            "Graph nodes and feature rows differ."
        )

    if list(graph.nodes()) != list(features_df.index):
        raise ValueError(
            "Graph node order and feature row order differ."
        )


def check_all_graphs_alignment(
    graphs,
    features_df,
    features_tensor
):
    expected_nodes = list(features_df.index)

    for graph_id, graph in enumerate(graphs, start=1):
        check_graph_feature_alignment(
            graph,
            features_df,
            features_tensor
        )

        if list(graph.nodes()) != expected_nodes:
            raise ValueError(
                f"Graph {graph_id} has an incorrect node order."
            )

    print(
        f"All {len(graphs)} directed graphs are aligned "
        "with the outgoing ARD."
    )

In [ ]:
# ------------------------------------------------------------
# 6. Convert directed NetworkX graphs to PyG
# ------------------------------------------------------------

def directed_graph_to_pyg_data(
    graph,
    features_tensor
):
    if not graph.is_directed():
        raise TypeError(
            "A directed graph is required."
        )

    if graph.number_of_nodes() != features_tensor.shape[0]:
        raise ValueError(
            "Graph nodes and feature rows differ."
        )

    data = from_networkx(graph)

    # For a NetworkX DiGraph, from_networkx preserves edge direction:
    #
    # edge_index[0] = source = citing paper
    # edge_index[1] = target = cited paper
    #
    # Therefore the PyG representation keeps:
    # citing -> cited.
    data.x = features_tensor.clone()

    if data.edge_index.shape[1] != graph.number_of_edges():
        raise ValueError(
            "The PyG edge count differs from the NetworkX edge count."
        )

    return data


def directed_graphs_to_pyg_data_list(
    graphs,
    features_tensor
):
    data_list = []

    for graph in tqdm(
        graphs,
        desc="Converting directed graphs to PyG"
    ):
        data_list.append(
            directed_graph_to_pyg_data(
                graph,
                features_tensor
            )
        )

    return data_list


In [ ]:
# ------------------------------------------------------------
# 7. Directed GCN encoder
# ------------------------------------------------------------

class DirectedGCNEncoder(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        embedding_dim,
        dropout=0.0
    ):
        super().__init__()

        self.conv_in = GCNConv(
            in_channels,
            embedding_dim
        )

        self.conv_out = GCNConv(
            in_channels,
            embedding_dim
        )

        self.combine = torch.nn.Linear(
            2 * embedding_dim,
            embedding_dim
        )

        self.dropout = float(dropout)

    def forward(
        self,
        x,
        edge_index,
        batch=None
    ):
        # Incoming information:
        # nodes that cite the current node.
        h_in = self.conv_in(
            x,
            edge_index
        )

        # Outgoing information:
        # nodes cited by the current node.
        reverse_edge_index = (
            edge_index.flip(0)
        )

        h_out = self.conv_out(
            x,
            reverse_edge_index
        )

        h = torch.cat(
            [h_in, h_out],
            dim=-1
        )

        h = self.combine(h)
        h = F.relu(h)

        if self.dropout > 0:
            h = F.dropout(
                h,
                p=self.dropout,
                training=self.training
            )

        return h

In [ ]:
# ------------------------------------------------------------
# 8. Directed DGI model
# ------------------------------------------------------------

class DirectedDeepGraphInfomax(torch.nn.Module):
    def __init__(
        self,
        in_channels,
        embedding_dim,
        dropout=0.0
    ):
        super().__init__()

        self.embedding_dim = embedding_dim

        self.encoder = DirectedGCNEncoder(
            in_channels=in_channels,
            embedding_dim=embedding_dim,
            dropout=dropout
        )

        self.discriminator_weight = (
            torch.nn.Parameter(
                torch.empty(
                    embedding_dim,
                    embedding_dim
                )
            )
        )

        self.reset_parameters()

    def reset_parameters(self):
        self.encoder.conv_in.reset_parameters()
        self.encoder.conv_out.reset_parameters()
        self.encoder.combine.reset_parameters()

        torch.nn.init.xavier_uniform_(
            self.discriminator_weight
        )

    @staticmethod
    def corrupt_features(
        x,
        batch=None
    ):
        if batch is None:
            permutation = torch.randperm(
                x.size(0),
                device=x.device
            )

            return x[permutation]

        corrupted_x = x.clone()

        graph_ids = torch.unique(
            batch,
            sorted=True
        )

        for graph_id in graph_ids:
            node_positions = torch.nonzero(
                batch == graph_id,
                as_tuple=False
            ).view(-1)

            local_permutation = (
                node_positions[
                    torch.randperm(
                        node_positions.numel(),
                        device=x.device
                    )
                ]
            )

            corrupted_x[node_positions] = (
                x[local_permutation]
            )

        return corrupted_x

    @staticmethod
    def readout(
        embeddings,
        batch=None
    ):
        if batch is None:
            return torch.sigmoid(
                embeddings.mean(
                    dim=0
                )
            )

        graph_summaries = global_mean_pool(
            embeddings,
            batch
        )

        return torch.sigmoid(
            graph_summaries
        )

    def discriminate(
        self,
        embeddings,
        summaries,
        batch=None
    ):
        projected_embeddings = (
            embeddings
            @ self.discriminator_weight
        )

        if batch is None:
            return (
                projected_embeddings
                * summaries
            ).sum(dim=-1)

        node_summaries = summaries[batch]

        return (
            projected_embeddings
            * node_summaries
        ).sum(dim=-1)

    def forward(
        self,
        x,
        edge_index,
        batch=None
    ):
        positive_embeddings = self.encoder(
            x,
            edge_index,
            batch
        )

        corrupted_x = self.corrupt_features(
            x,
            batch
        )

        negative_embeddings = self.encoder(
            corrupted_x,
            edge_index,
            batch
        )

        summaries = self.readout(
            positive_embeddings,
            batch
        )

        return (
            positive_embeddings,
            negative_embeddings,
            summaries
        )

    def loss(
        self,
        positive_embeddings,
        negative_embeddings,
        summaries,
        batch=None
    ):
        positive_logits = self.discriminate(
            positive_embeddings,
            summaries,
            batch
        )

        negative_logits = self.discriminate(
            negative_embeddings,
            summaries,
            batch
        )

        positive_loss = (
            F.binary_cross_entropy_with_logits(
                positive_logits,
                torch.ones_like(
                    positive_logits
                )
            )
        )

        negative_loss = (
            F.binary_cross_entropy_with_logits(
                negative_logits,
                torch.zeros_like(
                    negative_logits
                )
            )
        )

        return positive_loss + negative_loss

In [ ]:
# ------------------------------------------------------------
# 9. Model and training utilities
# ------------------------------------------------------------

def build_directed_dgi_model(
    num_features,
    embedding_dim,
    dropout=0.0
):
    return DirectedDeepGraphInfomax(
        in_channels=num_features,
        embedding_dim=embedding_dim,
        dropout=dropout
    )


def update_early_stopping(
    loss_value,
    model,
    best_loss,
    patience_counter
):
    if loss_value < best_loss:
        return (
            loss_value,
            0,
            copy.deepcopy(
                model.state_dict()
            )
        )

    return (
        best_loss,
        patience_counter + 1,
        None
    )


def extract_embeddings(
    model,
    data,
    device
):
    model.eval()
    data = data.to(device)

    with torch.no_grad():
        embeddings = model.encoder(
            data.x,
            data.edge_index,
            getattr(data, "batch", None)
        )

    return embeddings.cpu().numpy()

In [ ]:
# ------------------------------------------------------------
# 10. Train DGI on one directed graph
# ------------------------------------------------------------

def train_dgi_single_directed_graph(
    graph,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    dropout=0.0,
    seed=123,
    verbose=True
):
    if seed is not None:
        set_seed(seed)

    device = get_device()

    data = directed_graph_to_pyg_data(
        graph,
        features_tensor
    ).to(device)

    model = build_directed_dgi_model(
        num_features=data.num_features,
        embedding_dim=embedding_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_loss = float("inf")
    best_state = None
    patience_counter = 0
    loss_history = []

    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()

        (
            positive_embeddings,
            negative_embeddings,
            summaries
        ) = model(
            data.x,
            data.edge_index,
            batch=None
        )

        loss = model.loss(
            positive_embeddings,
            negative_embeddings,
            summaries,
            batch=None
        )

        loss.backward()
        optimizer.step()

        loss_value = float(loss.item())
        loss_history.append(loss_value)

        (
            new_best_loss,
            new_patience_counter,
            possible_best_state
        ) = update_early_stopping(
            loss_value,
            model,
            best_loss,
            patience_counter
        )

        best_loss = new_best_loss
        patience_counter = new_patience_counter

        if possible_best_state is not None:
            best_state = possible_best_state

        if verbose and epoch % 50 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Loss: {loss_value:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}"
                )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    embeddings = extract_embeddings(
        model,
        data,
        device
    )

    return (
        embeddings,
        loss_history,
        model
    )

In [ ]:
# ------------------------------------------------------------
# 11. METHOD 1: Multiple independent encoders
# ------------------------------------------------------------

def train_multiple_directed_encoders(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    dropout=0.0,
    seed=123,
    verbose=False
):
    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError(
            "No directed graphs were provided."
        )

    n_nodes = features_tensor.shape[0]

    embeddings_sum = np.zeros(
        (n_nodes, embedding_dim),
        dtype=np.float32
    )

    loss_histories = []

    for graph_id, graph in enumerate(
        tqdm(
            graphs,
            desc="Multiple directed encoders"
        ),
        start=1
    ):
        if verbose:
            print(
                f"\nGraph {graph_id}/{len(graphs)}"
            )

        (
            embeddings,
            loss_history,
            _
        ) = train_dgi_single_directed_graph(
            graph=graph,
            features_tensor=features_tensor,
            embedding_dim=embedding_dim,
            lr=lr,
            max_epochs=max_epochs,
            patience=patience,
            dropout=dropout,
            seed=None,
            verbose=verbose
        )

        embeddings_sum += embeddings
        loss_histories.append(
            loss_history
        )

    mean_embeddings = (
        embeddings_sum
        / len(graphs)
    )

    return (
        mean_embeddings,
        loss_histories
    )

In [ ]:
# ------------------------------------------------------------
# 12. METHOD 2: One shared sequential encoder
#     + training-time embeddings
# ------------------------------------------------------------

def train_one_directed_encoder_sequentially(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    dropout=0.0,
    seed=123,
    verbose=True
):
    """
    One Encoder methodology.

    Two embedding outputs are returned:

    1) post_training_mean_embeddings:
       after training, the best shared encoder is applied again to
       every reconstructed graph and embeddings are averaged by node.

    2) training_time_mean_embeddings:
       the positive node embeddings produced directly DURING the
       epoch with the lowest mean training loss are averaged by node
       across reconstructed graphs.

    Important:
    the encoder is updated after every graph, so training-time
    embeddings from different graphs within the same epoch come from
    successive states of the same shared encoder.
    """

    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError(
            "No directed graphs were provided."
        )

    device = get_device()

    data_list = directed_graphs_to_pyg_data_list(
        graphs,
        features_tensor
    )

    model = build_directed_dgi_model(
        num_features=data_list[0].num_features,
        embedding_dim=embedding_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_loss = float("inf")
    best_state = None
    best_training_time_mean_embeddings = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    for epoch in range(max_epochs):
        model.train()

        graph_indices = np.random.permutation(
            len(data_list)
        )

        epoch_losses = []

        # Keep embeddings by original graph index, even though
        # graph order is shuffled during training.
        epoch_training_embeddings = [
            None
        ] * len(data_list)

        for graph_index in graph_indices:
            data = data_list[
                graph_index
            ].to(device)

            optimizer.zero_grad()

            (
                positive_embeddings,
                negative_embeddings,
                summaries
            ) = model(
                data.x,
                data.edge_index,
                batch=None
            )

            # Save the embeddings produced DURING training,
            # before optimizer.step() updates the encoder.
            epoch_training_embeddings[
                graph_index
            ] = (
                positive_embeddings
                .detach()
                .cpu()
                .numpy()
                .copy()
            )

            loss = model.loss(
                positive_embeddings,
                negative_embeddings,
                summaries,
                batch=None
            )

            loss.backward()
            optimizer.step()

            epoch_losses.append(
                float(loss.item())
            )

        epoch_mean_loss = float(
            np.mean(epoch_losses)
        )

        epoch_loss_history.append(
            epoch_mean_loss
        )

        if epoch_mean_loss < best_loss:
            best_loss = epoch_mean_loss
            patience_counter = 0
            best_training_epoch = epoch

            # Best state follows the original implementation:
            # it is saved after all updates of the epoch.
            best_state = copy.deepcopy(
                model.state_dict()
            )

            if any(
                embedding is None
                for embedding in epoch_training_embeddings
            ):
                raise RuntimeError(
                    "Missing training-time embeddings for at least one graph."
                )

            best_training_time_mean_embeddings = np.mean(
                np.stack(
                    epoch_training_embeddings,
                    axis=0
                ),
                axis=0
            ).astype(
                np.float32
            )

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                "Mean loss across directed graphs: "
                f"{epoch_mean_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}"
                )
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    if best_training_time_mean_embeddings is None:
        raise RuntimeError(
            "No One Encoder training-time embeddings were retained."
        )

    # --------------------------------------------------------
    # Current / post-training representation
    # --------------------------------------------------------
    embeddings_sum = np.zeros(
        (
            features_tensor.shape[0],
            embedding_dim
        ),
        dtype=np.float32
    )

    model.eval()

    for data in tqdm(
        data_list,
        desc="Extracting One Encoder post-training embeddings"
    ):
        embeddings_sum += extract_embeddings(
            model,
            data,
            device
        )

    post_training_mean_embeddings = (
        embeddings_sum
        / len(data_list)
    )

    return (
        post_training_mean_embeddings,
        epoch_loss_history,
        model,
        best_training_time_mean_embeddings,
        best_training_epoch,
        best_loss
    )


In [ ]:
# ------------------------------------------------------------
# 13. METHOD 3: Shared FULL-BATCH encoder
#     + training-time embeddings
# ------------------------------------------------------------

def mean_node_embeddings_from_full_batch(
    positive_embeddings,
    batch_vector,
    number_of_nodes
):
    """
    Split PyG full-batch embeddings into graph-level matrices and
    average corresponding node embeddings across graph realizations.

    Every reconstructed Cora graph contains the same nodes in the same
    node order.
    """

    graph_embeddings = []

    graph_ids = torch.unique(
        batch_vector,
        sorted=True
    )

    for graph_id in graph_ids:
        graph_mask = (
            batch_vector == graph_id
        )

        graph_embedding = (
            positive_embeddings[
                graph_mask
            ]
        )

        if (
            graph_embedding.shape[0]
            != number_of_nodes
        ):
            raise ValueError(
                "A reconstructed graph in the full batch "
                "does not contain the expected number of nodes."
            )

        graph_embeddings.append(
            graph_embedding
            .detach()
            .cpu()
            .numpy()
            .copy()
        )

    return np.mean(
        np.stack(
            graph_embeddings,
            axis=0
        ),
        axis=0
    ).astype(
        np.float32
    )


def train_directed_encoder_full_batch(
    graphs,
    features_tensor,
    embedding_dim=64,
    lr=0.001,
    max_epochs=500,
    patience=20,
    dropout=0.0,
    seed=123,
    verbose=True
):
    """
    Train one shared directed DGI model using ALL reconstructed
    graphs in one full batch.

    Two embedding outputs are returned:

    1) post_training_mean_embeddings:
       after training, the best shared encoder is applied separately
       to every reconstructed graph and embeddings are averaged.

    2) training_time_mean_embeddings:
       the positive embeddings produced directly during the full-batch
       forward pass of the epoch with the lowest training loss are
       averaged by node across graph realizations.

    There is exactly one optimizer step per epoch.
    """

    set_seed(seed)

    if len(graphs) == 0:
        raise ValueError(
            "No directed graphs were provided."
        )

    device = get_device()

    data_list = directed_graphs_to_pyg_data_list(
        graphs,
        features_tensor
    )

    loader = DataLoader(
        data_list,
        batch_size=len(data_list),
        shuffle=False
    )

    if len(loader) != 1:
        raise RuntimeError(
            "Full-batch training should create exactly one batch."
        )

    model = build_directed_dgi_model(
        num_features=data_list[0].num_features,
        embedding_dim=embedding_dim,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    best_loss = float("inf")
    best_state = None
    best_training_time_mean_embeddings = None
    best_training_epoch = None

    patience_counter = 0
    epoch_loss_history = []

    if verbose:
        print(
            "Method 3 FULL BATCH:",
            len(data_list),
            "reconstructed graphs per optimization step."
        )
        print(
            "Optimizer updates per epoch: 1"
        )

    for epoch in range(max_epochs):
        model.train()

        batch = next(
            iter(loader)
        ).to(device)

        optimizer.zero_grad()

        (
            positive_embeddings,
            negative_embeddings,
            summaries
        ) = model(
            batch.x,
            batch.edge_index,
            batch=batch.batch
        )

        # Save the training-time representation BEFORE the
        # optimization step changes the encoder parameters.
        epoch_training_time_mean_embeddings = (
            mean_node_embeddings_from_full_batch(
                positive_embeddings=positive_embeddings,
                batch_vector=batch.batch,
                number_of_nodes=features_tensor.shape[0]
            )
        )

        loss = model.loss(
            positive_embeddings,
            negative_embeddings,
            summaries,
            batch=batch.batch
        )

        loss.backward()
        optimizer.step()

        epoch_loss = float(
            loss.item()
        )

        epoch_loss_history.append(
            epoch_loss
        )

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            patience_counter = 0
            best_training_epoch = epoch

            # Keep the same post-training convention as before:
            # state after the update associated with this best loss.
            best_state = copy.deepcopy(
                model.state_dict()
            )

            best_training_time_mean_embeddings = (
                epoch_training_time_mean_embeddings.copy()
            )

        else:
            patience_counter += 1

        if verbose and epoch % 10 == 0:
            print(
                f"Epoch {epoch:03d} | "
                f"Full-batch loss: {epoch_loss:.4f}"
            )

        if patience_counter >= patience:
            if verbose:
                print(
                    f"Early stopping at epoch {epoch}"
                )
            break

    if best_state is not None:
        model.load_state_dict(
            best_state
        )

    if best_training_time_mean_embeddings is None:
        raise RuntimeError(
            "No Full-Batch training-time embeddings were retained."
        )

    # --------------------------------------------------------
    # Current / post-training representation
    # --------------------------------------------------------
    embeddings_sum = np.zeros(
        (
            features_tensor.shape[0],
            embedding_dim
        ),
        dtype=np.float32
    )

    model.eval()

    for data in tqdm(
        data_list,
        desc="Extracting Full-Batch post-training embeddings"
    ):
        embeddings_sum += extract_embeddings(
            model,
            data,
            device
        )

    post_training_mean_embeddings = (
        embeddings_sum
        / len(data_list)
    )

    return (
        post_training_mean_embeddings,
        epoch_loss_history,
        model,
        best_training_time_mean_embeddings,
        best_training_epoch,
        best_loss
    )


In [ ]:
# ------------------------------------------------------------
# 14. Save utilities
# ------------------------------------------------------------

def save_embeddings(
    embeddings,
    output_path
):
    np.save(
        output_path,
        embeddings
    )


def save_pickle(
    value,
    output_path
):
    with open(output_path, "wb") as file:
        pickle.dump(value, file)


def save_model_state(
    model,
    output_path
):
    torch.save(
        model.state_dict(),
        output_path
    )

In [ ]:
# ------------------------------------------------------------
# 15. Load Step 2 output and directed ARD
# ------------------------------------------------------------

features_df, features_tensor = (
    load_outgoing_ard_features(
        ard_path
    )
)

node_order = (
    features_df.index.tolist()
)

graphs = load_augmented_directed_graphs(
    graphs_path
)

graphs = enforce_directed_node_order_all(
    graphs,
    node_order
)

original_graph = (
    load_original_cora_directed_graph(
        original_edges_path,
        labels_path,
        node_order
    )
)

original_graph = enforce_directed_node_order(
    original_graph,
    node_order
)

check_all_graphs_alignment(
    graphs,
    features_df,
    features_tensor
)

check_graph_feature_alignment(
    original_graph,
    features_df,
    features_tensor
)

# ------------------------------------------------------------
# Critical citation-orientation validation
# ------------------------------------------------------------
# Since the seven Cora classes are mutually exclusive and exhaustive:
#
#   sum_k ARD_out[i,k] = out_degree(i)
#
# for every node i.
#
# If the original citation direction were reversed, this check would fail.
ard_row_sums = (
    features_df.sum(axis=1).astype(int)
)

true_out_degrees = pd.Series(
    dict(original_graph.out_degree()),
    dtype=int
).reindex(features_df.index)

orientation_mismatch = (
    ard_row_sums != true_out_degrees
)

if orientation_mismatch.any():
    mismatch_nodes = orientation_mismatch[
        orientation_mismatch
    ].index.tolist()

    example_nodes = mismatch_nodes[:10]

    comparison = pd.DataFrame({
        "ard_row_sum": ard_row_sums.loc[example_nodes],
        "original_out_degree": true_out_degrees.loc[example_nodes]
    })

    print(comparison)

    raise ValueError(
        "Outgoing ARD row sums do not match the out-degrees "
        "of the original Cora graph. Check citation orientation."
    )

if int(ard_row_sums.sum()) != original_graph.number_of_edges():
    raise ValueError(
        "Total outgoing ARD counts do not match the number "
        "of directed edges in the original graph."
    )

print("\n=== Directed input summary ===")
print("Synthetic graphs:", len(graphs))
print("Nodes:", len(node_order))
print("Outgoing ARD shape:", features_tensor.shape)
print(
    "Original directed edges:",
    original_graph.number_of_edges()
)
print(
    "Example corrected synthetic edges:",
    graphs[0].number_of_edges()
)
print(
    "Original graph is directed:",
    original_graph.is_directed()
)
print(
    "Synthetic graphs are directed:",
    all(graph.is_directed() for graph in graphs)
)
print(
    "Orientation check passed: "
    "ARD row sums = original out-degrees."
)
print(
    "Citation convention confirmed: citing -> cited."
)


In [ ]:
# ------------------------------------------------------------
# 16. Experiment configuration
# ------------------------------------------------------------

EMBEDDING_DIM = 64
LEARNING_RATE = 0.001
MAX_EPOCHS = 500
PATIENCE = 20
DROPOUT = 0.0
SEED = 123

# METHOD 3 uses FULL BATCH:
# every reconstructed graph is included in the same optimization step.
FULL_BATCH_NUMBER_OF_GRAPHS = len(graphs)

configuration = {
    "embedding_dim": EMBEDDING_DIM,
    "learning_rate": LEARNING_RATE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "dropout": DROPOUT,
    "seed": SEED,
    "batch_mode": "full_batch",
    "graphs_per_batch": FULL_BATCH_NUMBER_OF_GRAPHS,
    "optimizer_steps_per_epoch_method3": 1,
    "save_training_time_embeddings": True,
    "training_time_selection": "epoch_with_lowest_training_loss",
    "number_of_synthetic_graphs": len(graphs),
    "number_of_nodes": len(node_order),
    "number_of_features": features_tensor.shape[1],
    "graph_file": str(graphs_path),
    "feature_file": str(ard_path),
    "encoder": (
        "DirectedGCNEncoder with incoming and outgoing branches"
    )
}

with open(
    output_folder / "directed_dgi_configuration.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        configuration,
        file,
        indent=2
    )

pd.Series(
    node_order,
    name="node_id"
).to_csv(
    output_folder / "node_order.csv",
    index=False
)

print(configuration)


In [ ]:
# ------------------------------------------------------------
# 17. METHOD 1: Multiple Encoders
# ------------------------------------------------------------

(
    embeddings_method1,
    loss_histories_method1
) = train_multiple_directed_encoders(
    graphs=graphs,
    features_tensor=features_tensor,
    embedding_dim=EMBEDDING_DIM,
    lr=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    dropout=DROPOUT,
    seed=SEED,
    verbose=False
)

print(
    "Method 1 embeddings:",
    embeddings_method1.shape
)

save_embeddings(
    embeddings_method1,
    output_folder
    / "embeddings_method1_multiple_encoders_directed_ard_out.npy"
)

save_pickle(
    loss_histories_method1,
    output_folder
    / "loss_histories_method1_multiple_encoders_directed_ard_out.pkl"
)

In [ ]:
# ------------------------------------------------------------
# 18. METHOD 2: One Encoder
# ------------------------------------------------------------

(
    embeddings_method2,
    loss_history_method2,
    model_method2,
    embeddings_method2_training,
    method2_training_best_epoch,
    method2_training_best_loss
) = train_one_directed_encoder_sequentially(
    graphs=graphs,
    features_tensor=features_tensor,
    embedding_dim=EMBEDDING_DIM,
    lr=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    dropout=DROPOUT,
    seed=SEED,
    verbose=True
)

print(
    "Method 2 post-training embeddings:",
    embeddings_method2.shape
)

print(
    "Method 2 training-time embeddings:",
    embeddings_method2_training.shape
)

print(
    "Method 2 best training epoch:",
    method2_training_best_epoch
)

print(
    "Method 2 best training loss:",
    method2_training_best_loss
)

# Current/post-training output
save_embeddings(
    embeddings_method2,
    output_folder
    / "embeddings_method2_one_encoder_directed_ard_out.npy"
)

# New training-time output
save_embeddings(
    embeddings_method2_training,
    output_folder
    / "embeddings_method2_one_encoder_TRAINING_MEAN_directed_ard_out.npy"
)

save_pickle(
    loss_history_method2,
    output_folder
    / "loss_history_method2_one_encoder_directed_ard_out.pkl"
)

save_model_state(
    model_method2,
    output_folder
    / "model_method2_one_encoder_directed_ard_out.pt"
)


In [ ]:
# ------------------------------------------------------------
# 19. METHOD 3: FULL BATCH
# ------------------------------------------------------------

(
    embeddings_method3,
    loss_history_method3,
    model_method3,
    embeddings_method3_training,
    method3_training_best_epoch,
    method3_training_best_loss
) = train_directed_encoder_full_batch(
    graphs=graphs,
    features_tensor=features_tensor,
    embedding_dim=EMBEDDING_DIM,
    lr=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    dropout=DROPOUT,
    seed=SEED,
    verbose=True
)

print(
    "Method 3 post-training embeddings:",
    embeddings_method3.shape
)

print(
    "Method 3 training-time embeddings:",
    embeddings_method3_training.shape
)

print(
    "Method 3 best training epoch:",
    method3_training_best_epoch
)

print(
    "Method 3 best training loss:",
    method3_training_best_loss
)

# Current/post-training output
save_embeddings(
    embeddings_method3,
    output_folder
    / "embeddings_method3_batch_directed_ard_out.npy"
)

# New training-time output
save_embeddings(
    embeddings_method3_training,
    output_folder
    / "embeddings_method3_batch_TRAINING_MEAN_directed_ard_out.npy"
)

save_pickle(
    loss_history_method3,
    output_folder
    / "loss_history_method3_batch_directed_ard_out.pkl"
)

save_model_state(
    model_method3,
    output_folder
    / "model_method3_batch_directed_ard_out.pt"
)


In [ ]:
# ------------------------------------------------------------
# 20. Original directed Cora graph
# ------------------------------------------------------------

(
    embeddings_original,
    loss_history_original,
    model_original
) = train_dgi_single_directed_graph(
    graph=original_graph,
    features_tensor=features_tensor,
    embedding_dim=EMBEDDING_DIM,
    lr=LEARNING_RATE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    dropout=DROPOUT,
    seed=SEED,
    verbose=True
)

print(
    "Original directed graph embeddings:",
    embeddings_original.shape
)

save_embeddings(
    embeddings_original,
    output_folder
    / "embeddings_original_directed_ard_out.npy"
)

save_pickle(
    loss_history_original,
    output_folder
    / "loss_history_original_directed_ard_out.pkl"
)

save_model_state(
    model_original,
    output_folder
    / "model_original_directed_ard_out.pt"
)

In [ ]:
# ------------------------------------------------------------
# 21. Compatibility copies for Step 4
# ------------------------------------------------------------

save_embeddings(
    embeddings_original,
    output_folder
    / "embeddings_original.npy"
)

save_embeddings(
    embeddings_method1,
    output_folder
    / "embeddings_ensemble_mean.npy"
)

save_embeddings(
    embeddings_method2,
    output_folder
    / "embeddings_shared_dgi_mean.npy"
)

save_embeddings(
    embeddings_method3,
    output_folder
    / "embeddings_batch_mean.npy"
)

print(
    "Compatibility embedding files saved for Step 4."
)

In [ ]:
# ------------------------------------------------------------
# 22. Optional structure-only input
# ------------------------------------------------------------

RUN_STRUCTURE_ONLY = False

if RUN_STRUCTURE_ONLY:
    number_of_nodes = len(node_order)

    structure_features_array = np.eye(
        number_of_nodes,
        dtype=np.float32
    )

    structure_features_tensor = torch.from_numpy(
        structure_features_array
    )

    structure_output_folder = (
        output_folder
        / "structure_only"
    )

    structure_output_folder.mkdir(
        parents=True,
        exist_ok=True
    )

    print(
        "Structure-only input:",
        structure_features_tensor.shape
    )
else:
    print(
        "Structure-only experiment is disabled."
    )

In [ ]:
# ------------------------------------------------------------
# 23. Optional METHOD 1 with graph structure only
# ------------------------------------------------------------

if RUN_STRUCTURE_ONLY:
    (
        embeddings_structure_method1,
        loss_structure_method1
    ) = train_multiple_directed_encoders(
        graphs=graphs,
        features_tensor=structure_features_tensor,
        embedding_dim=EMBEDDING_DIM,
        lr=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        dropout=DROPOUT,
        seed=SEED,
        verbose=False
    )

    save_embeddings(
        embeddings_structure_method1,
        structure_output_folder
        / "embeddings_method1_multiple_encoders_structure_only.npy"
    )

    save_pickle(
        loss_structure_method1,
        structure_output_folder
        / "loss_method1_multiple_encoders_structure_only.pkl"
    )

In [ ]:
# ------------------------------------------------------------
# 24. Optional METHOD 2 with graph structure only
# ------------------------------------------------------------

if RUN_STRUCTURE_ONLY:
    (
        embeddings_structure_method2,
        loss_structure_method2,
        model_structure_method2,
        embeddings_structure_method2_training,
        structure_method2_best_epoch,
        structure_method2_best_loss
    ) = train_one_directed_encoder_sequentially(
        graphs=graphs,
        features_tensor=structure_features_tensor,
        embedding_dim=EMBEDDING_DIM,
        lr=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        dropout=DROPOUT,
        seed=SEED,
        verbose=True
    )

    save_embeddings(
        embeddings_structure_method2,
        structure_output_folder
        / "embeddings_method2_one_encoder_structure_only.npy"
    )

    save_embeddings(
        embeddings_structure_method2_training,
        structure_output_folder
        / "embeddings_method2_one_encoder_TRAINING_MEAN_structure_only.npy"
    )

    save_pickle(
        loss_structure_method2,
        structure_output_folder
        / "loss_method2_one_encoder_structure_only.pkl"
    )

    save_model_state(
        model_structure_method2,
        structure_output_folder
        / "model_method2_one_encoder_structure_only.pt"
    )


In [ ]:
# ------------------------------------------------------------
# 25. Optional METHOD 3 with graph structure only — FULL BATCH
# ------------------------------------------------------------

if RUN_STRUCTURE_ONLY:
    (
        embeddings_structure_method3,
        loss_structure_method3,
        model_structure_method3,
        embeddings_structure_method3_training,
        structure_method3_best_epoch,
        structure_method3_best_loss
    ) = train_directed_encoder_full_batch(
        graphs=graphs,
        features_tensor=structure_features_tensor,
        embedding_dim=EMBEDDING_DIM,
        lr=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        dropout=DROPOUT,
        seed=SEED,
        verbose=True
    )

    save_embeddings(
        embeddings_structure_method3,
        structure_output_folder
        / "embeddings_method3_batch_structure_only.npy"
    )

    save_embeddings(
        embeddings_structure_method3_training,
        structure_output_folder
        / "embeddings_method3_batch_TRAINING_MEAN_structure_only.npy"
    )

    save_pickle(
        loss_structure_method3,
        structure_output_folder
        / "loss_method3_batch_structure_only.pkl"
    )

    save_model_state(
        model_structure_method3,
        structure_output_folder
        / "model_method3_batch_structure_only.pt"
    )


In [ ]:
# ------------------------------------------------------------
# 26. Optional original directed graph with structure only
# ------------------------------------------------------------

if RUN_STRUCTURE_ONLY:
    (
        embeddings_original_structure,
        loss_original_structure,
        model_original_structure
    ) = train_dgi_single_directed_graph(
        graph=original_graph,
        features_tensor=structure_features_tensor,
        embedding_dim=EMBEDDING_DIM,
        lr=LEARNING_RATE,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        dropout=DROPOUT,
        seed=SEED,
        verbose=True
    )

    save_embeddings(
        embeddings_original_structure,
        structure_output_folder
        / "embeddings_original_directed_structure_only.npy"
    )

    save_pickle(
        loss_original_structure,
        structure_output_folder
        / "loss_original_directed_structure_only.pkl"
    )

    save_model_state(
        model_original_structure,
        structure_output_folder
        / "model_original_directed_structure_only.pt"
    )

In [ ]:
# ------------------------------------------------------------
# 27. Final checks
# ------------------------------------------------------------

expected_shape = (
    len(node_order),
    EMBEDDING_DIM
)

embedding_outputs_to_validate = {
    "original": embeddings_original,
    "method1": embeddings_method1,
    "method2_post_training": embeddings_method2,
    "method2_training_time": embeddings_method2_training,
    "method3_post_training": embeddings_method3,
    "method3_training_time": embeddings_method3_training,
}

for embedding_name, embeddings in (
    embedding_outputs_to_validate.items()
):
    if embeddings.shape != expected_shape:
        raise ValueError(
            f"{embedding_name} embeddings have shape "
            f"{embeddings.shape}; expected {expected_shape}."
        )

    if not np.isfinite(embeddings).all():
        raise ValueError(
            f"{embedding_name} embeddings contain NaN or infinity."
        )

print("Step 3 completed successfully.")
print("Embedding shape:", expected_shape)
print("Citation orientation: citing -> cited")
print(
    "Outgoing ARD validated against the original graph out-degrees."
)
print(
    "Training-time embeddings saved for One Encoder and Full Batch."
)
print("Outputs saved in:", output_folder)

print("\nNew files:")
print(
    "embeddings_method2_one_encoder_TRAINING_MEAN_directed_ard_out.npy"
)
print(
    "embeddings_method3_batch_TRAINING_MEAN_directed_ard_out.npy"
)
